In [1]:
import pandas as pd
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt


import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [2]:
def gather_data(first_year, last_year):
    years = []
    for i in range(first_year, last_year+1):
        months = []
        for j in range(1,4):
            months.append(pd.read_csv(f"data/PRICE_AND_DEMAND_{i}{j:02}_VIC1.csv"))
        years.append(pd.concat(months))
    data = pd.concat(years)
    data["SETTLEMENTDATE"] = pd.to_datetime(data["SETTLEMENTDATE"])
    data["hour"] = data["SETTLEMENTDATE"].dt.hour       # hour of day
    data["minute"] = data["SETTLEMENTDATE"].dt.minute + 60*data["hour"]
    data["DoY"] = data["SETTLEMENTDATE"].dt.dayofyear   # day of year


    # take sin and cosine. maps time to a point on the unit circle essentially
    data["minute_sin"] = data["minute"].apply(lambda x: np.sin(x*2*np.pi/1440))  # sine of hour to capture daily swings
    data["minute_cos"] = data["minute"].apply(lambda x: np.cos(x*2*np.pi/1440))  # cosine of hour ""
    data["DoY_sin"] = data["DoY"].apply(lambda x: np.sin(x*2*np.pi/365))    # sine of DoY to capture seasonality
    data["DoY_cos"] = data["DoY"].apply(lambda x: np.cos(x*2*np.pi/365))    # cosine of DoY ""

    # drop unnecessary data
    data = data.drop(columns=["REGION", "PERIODTYPE", "hour"])

    return data.sort_values("SETTLEMENTDATE", ignore_index=True)

In [ ]:
def make_future_row(row, h):
    """
    given a row at time t, return row with features for time t+h
    """
    new_row = row.copy()
    
    # horizon
    new_row["h"] = h
    
    # future datetime
    future_time = row["SETTLEMENTDATE"] + pd.to_timedelta(5 * h, unit="m")
    
    # minute
    minute_future = future_time.hour * 60 + future_time.minute
    new_row["minute_future"] = minute_future
    new_row["minute_sin_future"] = np.sin(minute_future * 2 * np.pi / 1440)
    new_row["minute_cos_future"] = np.cos(minute_future * 2 * np.pi / 1440)
    
    # DoY
    doy_future = future_time.dayofyear
    new_row["DoY_future"] = doy_future
    new_row["DoY_sin_future"] = np.sin(doy_future * 2 * np.pi / 365)
    new_row["DoY_cos_future"] = np.cos(doy_future * 2 * np.pi / 365)
    
    return new_row

def add_horizons(df, horizons):
    df = df.sort_values("SETTLEMENTDATE").reset_index(drop=True)
    dfs = []
    
    for h in horizons:
        temp = df.copy()
        
        temp["target"] = temp["RRP"].shift(-h)
        temp["h"] = h
        
        temp["SETTLEMENTDATE_future"] = temp["SETTLEMENTDATE"] + pd.to_timedelta(5 * h, unit="m")
        
        hour_future = temp["SETTLEMENTDATE_future"].dt.hour
        minute_future = temp["SETTLEMENTDATE_future"].dt.minute + 60 * hour_future
        temp["minute_future"] = minute_future
        temp["minute_sin_future"] = np.sin(minute_future * 2 * np.pi / 1440)
        temp["minute_cos_future"] = np.cos(minute_future * 2 * np.pi / 1440)

        doy_future = temp["SETTLEMENTDATE_future"].dt.dayofyear
        temp["DoY_future"] = doy_future
        temp["DoY_sin_future"] = np.sin(doy_future * 2 * np.pi / 365)
        temp["DoY_cos_future"] = np.cos(doy_future * 2 * np.pi / 365)
        
        dfs.append(temp)
    
    df_multi = pd.concat(dfs, ignore_index=True)
    df_multi = df_multi.dropna(subset=["target"])
    

    return df_multi

In [26]:
price_data = gather_data(2025,2025)

In [78]:
horizons = range(1,289)

In [79]:
data = add_horizons(price_data, horizons)

In [47]:
def train_model(df: pd.DataFrame, split_date):
    drop_cols = ["target", "SETTLEMENTDATE", "SETTLEMENTDATE_future"]
    features = [c for c in df.columns if c not in drop_cols]


    train = df[df["SETTLEMENTDATE_future"] < split_date]
    test  = df[df["SETTLEMENTDATE_future"] >= split_date]

    X_train, y_train = train[features], train["target"]
    X_test,  y_test  = test[features],  test["target"]
    
    model = xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=50,
        eval_metric="mae",
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100,
    )


    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"MAE:  {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")

    return model, preds, y_test

In [80]:
model, preds, y_test = train_model(data, "2025-02-01")

[0]	validation_0-mae:63.00690
[83]	validation_0-mae:55.44357
MAE:  54.45
RMSE: 157.50


In [94]:
def plot_predictions(preds, y_test, max_val=1000):
    mask = y_test < max_val
    plt.figure(figsize=(8, 8))
    plt.scatter(preds[mask], y_test[mask], alpha=0.01, s=1)
    plt.scatter(preds[mask], preds[mask],alpha=0.01)
    plt.show()